In [57]:
import pandas as pd
import json

In [58]:
MY_AUTH_KEY = 'M2MwNDQ0OGItNTY2Zi00MTlhLWJjNDAtNmNlZmI1MzBhMmRjOjE1MWFjYmQxLWM4NGItNGQxOS1hNWFjLTM5OTFkOGUwMDM0Yw=='

In [59]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores.faiss import FAISS
from langchain_core.documents import Document

In [60]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [61]:
df = pd.read_csv('products.csv')
df = df.drop(['category', 'price'], axis=1)
df.to_csv('prods.txt', index=False, sep=' ')

In [62]:
df

,name
0,"Картофель в сетке ~2,3 кг"
1,Бананы
2,Бананы Global Village
3,Мандарины
4,Морковь
...,...
373,Семечки тыквенные Вкус & Польза очищенные 150 г
374,Семечки подсолнечные Mixbar жареные 200 г
375,Миндаль Mixbar жареный сладкий 100 г
376,Чипсы фруктовые Яблоков из кисло-сладких яблок...


In [63]:
with open('prods.txt', 'r', encoding='utf-8') as file:
    text = file.read()

# Создание объекта Document
doc = Document(page_content=text, metadata={})

In [64]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=30, chunk_overlap=5)
split_docs = text_splitter.split_documents([doc])

In [65]:
len(split_docs)

713

In [66]:
split_docs

[Document(metadata={}, page_content='name'),
 Document(metadata={}, page_content='"Картофель в сетке ~2,3 кг"'),
 Document(metadata={}, page_content='Бананы'),
 Document(metadata={}, page_content='"Бананы Global Village"'),
 Document(metadata={}, page_content='Мандарины\nМорковь'),
 Document(metadata={}, page_content='"Лук репчатый"\nКартофель'),
 Document(metadata={}, page_content='Лимоны'),
 Document(metadata={}, page_content='"Виноград белый без косточки"'),
 Document(metadata={}, page_content='Помидоры\n"Яблоки Ред Делишес"'),
 Document(metadata={}, page_content='"Яблоки Голден"'),
 Document(metadata={}, page_content='"Шампиньоны свежие Global'),
 Document(metadata={}, page_content='Village целые"'),
 Document(metadata={}, page_content='Свекла\n"Капуста Китайская"'),
 Document(metadata={}, page_content='"Салат листовой в горшочке'),
 Document(metadata={}, page_content='~150 г"'),
 Document(metadata={}, page_content='"Шампиньоны свежие Global'),
 Document(metadata={}, page_content='

In [67]:
%%time
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embedding = HuggingFaceEmbeddings(model_name=model_name,
                                  model_kwargs=model_kwargs,
                                  encode_kwargs=encode_kwargs)

vector_store = FAISS.from_documents(split_docs, embedding=embedding)

CPU times: user 10.1 s, sys: 727 ms, total: 10.8 s
Wall time: 6.08 s


In [68]:
for doc in split_docs:
    print(doc.page_content)

name
"Картофель в сетке ~2,3 кг"
Бананы
"Бананы Global Village"
Мандарины
Морковь
"Лук репчатый"
Картофель
Лимоны
"Виноград белый без косточки"
Помидоры
"Яблоки Ред Делишес"
"Яблоки Голден"
"Шампиньоны свежие Global
Village целые"
Свекла
"Капуста Китайская"
"Салат листовой в горшочке
~150 г"
"Шампиньоны свежие Global
Village целые 400 г"
"Помидоры красные Global
Village Черри мелкие 250 г"
Апельсины
"Перец сладкий красный"
Чеснок
"Салат Айсберг"
"Яблоки Гренни Смит"
"Набор зелени Global Village
Ассорти 70 г"
"Помидоры сливовидные
Фламенко"
"Виноград Ред Глоб Чили"
"Мандарины Выгодно в сетке"
"Огурец длинноплодный Global
Village"
"Овощная смесь Морозко Green
Овощное трио замороженная 400
400 г"
"Салат Фриллис 1 шт"
"Шампиньоны свежие Global
Village 250 г"
"Груши Конференция"
"Отварная свекла Global
Village целая очищенная 500
500 г"
"Кабачки ~900 г"
"Огурцы среднеплодные"
"Лук зеленый 50 г"
"Укроп Global Village 30 г"
"Клубника Красная цена
цена быстрозамороженная 300
300 г"
Киви
"Морко

In [69]:
embedding_retriever = vector_store.as_retriever(search_kwargs={"k": 100})

In [70]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
from langchain_gigachat import GigaChat
from langchain.chains import create_retrieval_chain

In [71]:
llm = GigaChat(credentials=MY_AUTH_KEY,
              model='GigaChat:latest',
               verify_ssl_certs=False,
               profanity_check=False)
prompt = ChatPromptTemplate.from_template('''Составь меню по запросу пользователя. \
Используй при этом только продукты из контекста. Если в контексте нет \
информации для ответа, сообщи об этом пользователю.
Контекст: {context}
Вопрос: {input}
Ответ:'''
)

In [72]:
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
    )
     

In [73]:
retrieval_chain = create_retrieval_chain(embedding_retriever, document_chain)


In [77]:
query = 'Составь вегетарианское меню на 3 дня. Животные жиры и рыба разрешается. Количество приемов пищи в день - 3. Выдай названия и описания блюд.'
response = retrieval_chain.invoke(
    {'input': query}
)
response

{'input': 'Составь вегетарианское меню на 3 дня. Животные жиры и рыба разрешается. Количество приемов пищи в день - 3. Выдай названия и описания блюд.',
 'context': [Document(id='520e7355-5e66-468d-8abe-f4cb86a228e1', metadata={}, page_content='обжаренные с овощами 350 г"'),
  Document(id='d1884bb8-656d-4095-9cc6-fcfd594af94d', metadata={}, page_content='Овощи гриль с'),
  Document(id='03d33b41-aa1a-40fa-8e6f-46a3b5311935', metadata={}, page_content='Diet маринованная 350 г"'),
  Document(id='c4c0e366-3e80-4662-b0bf-4160f880b353', metadata={}, page_content='обжаренные с овощами 510 г"'),
  Document(id='8ccf8550-ec02-416d-932c-ef57b6f22c2a', metadata={}, page_content='жареные с добавлением'),
  Document(id='3574f25e-ce25-49dc-b826-4ed660c020c0', metadata={}, page_content='из соленых овощей 400 г"'),
  Document(id='5a1fa495-2c84-45a4-8be0-c148c3f2bafa', metadata={}, page_content='"Баклажаны с овощами Пиканта'),
  Document(id='e04c6d36-68d5-4588-beff-9a11098177d5', metadata={}, page_conte

In [78]:
# Преобразование объектов Document в словари
response['context'] = [
    {
        'id': doc.id,
        'metadata': doc.metadata,
        'page_content': doc.page_content
    }
    for doc in response['context']
]

# Сохранение ответа в JSON файл
with open('response.json', 'w', encoding='utf-8') as json_file:
    json.dump(response, json_file, ensure_ascii=False, indent=4)